# Notebook 5 — Feature Engineering

## Objective

This notebook prepares the features required for the machine learning model that predicts whether an order will be delivered late or on time.

The feature engineering process is based on the findings from the EDA stage. The main objectives are to:

* Create predictive features using information available at prediction time.
* Avoid data leakage from future information.
* Handle missing values appropriately.
* Encode categorical variables.
* Scale numerical features if required by the selected model.
* Fit preprocessing transformations on the training data only.
* Apply the fitted transformations to the validation and test datasets.
* Save the fitted transformers and the final feature list for use in production.

### Input Data

The notebook uses the train, validation, and test datasets created in Notebook 3.

### Output Artifacts

The following artifacts will be generated:

* Final feature tables for train, validation, and test.
* Fitted preprocessing transformers.
* Feature list documenting the final model inputs.

## Load Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("../Artifacts/train.csv")
validation = pd.read_csv("../Artifacts/validation.csv")
test = pd.read_csv("../Artifacts/test.csv")

In [3]:
print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67533, 14)
Validation shape: (14471, 14)
Test shape: (14472, 14)


In [4]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'number_of_items',
 'total_price',
 'total_freight',
 'number_of_payments',
 'total_payment_value',
 'delivery_label']

## Feature Engineering

The goal of this notebook is to transform the original train, validation, and test datasets into model-ready features while avoiding data leakage.

The features are based on the patterns identified during the EDA stage. Only information that would be available at prediction time is used.

### Features to be created

* **Purchase hour:** hour of the day when the order was placed.
* **Purchase weekday:** day of the week when the order was placed.
* **Purchase month:** month in which the order was placed.
* **Estimated delivery days:** number of days between the purchase timestamp and the estimated delivery date.
* **Holiday indicator:** whether the order was placed on a Brazilian public holiday.

### Leakage prevention

Features that depend on events occurring after the prediction point, such as actual delivery duration or the actual customer delivery date, will not be used as model features.

All fitted preprocessing transformations will be learned using the training split only and then applied to the validation and test splits.


In [5]:
for df in [train, validation, test]:
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )
    
    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"]
    )

In [6]:
train[[
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]].dtypes

order_purchase_timestamp         datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [7]:
for df in [train, validation, test]:
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
    
    df["purchase_weekday"] = (
        df["order_purchase_timestamp"].dt.day_name()
    )
    
    df["purchase_month"] = (
        df["order_purchase_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

In [8]:
train[
    [
        "purchase_hour",
        "purchase_weekday",
        "purchase_month"
    ]
].head()

,purchase_hour,purchase_weekday,purchase_month
0,12,Thursday,2016-09
1,9,Monday,2016-10
2,16,Monday,2016-10
3,21,Monday,2016-10
4,21,Monday,2016-10


In [9]:
for df in [train, validation, test]:
    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 86400

In [10]:
train["estimated_delivery_days"].describe()

count    67533.000000
mean        24.597288
std          8.022133
min          7.005127
25%         19.543333
50%         23.621377
75%         28.565984
max        155.135463
Name: estimated_delivery_days, dtype: float64

### Estimated Delivery Time

The `estimated_delivery_days` feature represents the expected number of days between the order purchase timestamp and the estimated delivery date.

This feature is available at prediction time because the estimated delivery date is associated with the order when it is placed. Therefore, it can be used as a predictive feature without relying on the actual delivery outcome.

The distribution of the feature will be reviewed for missing values and potential outliers before preprocessing.

In [11]:
train["estimated_delivery_days"].isna().sum()

np.int64(0)

No missing values were found in `estimated_delivery_days` in the training data, so no imputation is required for this feature.


In [12]:
import holidays

br_holidays = holidays.Brazil(years=range(2016, 2019))

In [13]:
for df in [train, validation, test]:
    df["is_holiday"] = (
        df["order_purchase_timestamp"].dt.date.isin(br_holidays)
    )

In [14]:
train["is_holiday"].value_counts()

is_holiday
False    66462
True      1071
Name: count, dtype: int64

### Holiday Indicator

The `is_holiday` feature identifies whether an order was placed on a Brazilian public holiday.

The feature is derived from the order purchase date and does not use any information from the delivery outcome. Therefore, it is available at prediction time and does not introduce data leakage.

The training data contains 1,071 holiday orders and 66,462 non-holiday orders.

In [15]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'number_of_items',
 'total_price',
 'total_freight',
 'number_of_payments',
 'total_payment_value',
 'delivery_label',
 'purchase_hour',
 'purchase_weekday',
 'purchase_month',
 'estimated_delivery_days',
 'is_holiday']

### Geographic Features

The EDA identified customer geography and customer-seller distance as potentially useful predictive features.

These features are derived only from information available at prediction time:

* `customer_state` is obtained from the customer's location.
* `distance_km` is calculated from the customer and seller geographic coordinates.

The geographic source tables are loaded separately because these features were not included in the train, validation, and test artifacts created in the previous notebook.


In [16]:
customers = pd.read_csv("../Data/raw/olist_customers_dataset.csv")
sellers = pd.read_csv("../Data/raw/olist_sellers_dataset.csv")
geolocation = pd.read_csv("../Data/raw/olist_geolocation_dataset.csv")

In [17]:
print("Customers:", customers.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)

Customers: (99441, 5)
Sellers: (3095, 4)
Geolocation: (1000163, 5)


### Customer State

The `customer_state` feature represents the state where the customer is located.

Customer location is available at prediction time, so this feature can be used without introducing future information or data leakage.


In [18]:
train = train.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

validation = validation.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

test = test.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)

In [19]:
train["customer_state"].isna().sum()

np.int64(0)

The `customer_state` feature was successfully added to the train, validation, and test datasets. No missing customer states were found in the training data.


### Geographic Coordinates by ZIP Prefix

The geolocation dataset contains multiple coordinate observations for the same ZIP code prefix. To obtain a single representative location for each prefix, the mean latitude and longitude are calculated.

This aggregated lookup table will be used to derive customer and seller coordinates and calculate the geographical distance between them.


In [20]:
geo_by_zip = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)
geo_by_zip.head()

,geolocation_zip_code_prefix,latitude,longitude
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


### Customer Coordinates

Customer geographic coordinates are obtained by mapping each customer's ZIP code prefix to the corresponding aggregated geographic coordinates.

These coordinates are based on customer location information available at prediction time.


In [21]:
train = train.merge(
    customers[[
        "customer_id",
        "customer_zip_code_prefix"
    ]],
    on="customer_id",
    how="left"
)

validation = validation.merge(
    customers[[
        "customer_id",
        "customer_zip_code_prefix"
    ]],
    on="customer_id",
    how="left"
)

test = test.merge(
    customers[[
        "customer_id",
        "customer_zip_code_prefix"
    ]],
    on="customer_id",
    how="left"
)

In [22]:
train = train.merge(
    geo_by_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

validation = validation.merge(
    geo_by_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

test = test.merge(
    geo_by_zip,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

In [23]:
train[["customer_zip_code_prefix", "latitude", "longitude"]].isna().sum()

customer_zip_code_prefix      0
latitude                    181
longitude                   181
dtype: int64

In [24]:
train = train.rename(
    columns={
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

validation = validation.rename(
    columns={
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

test = test.rename(
    columns={
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

### Seller Information

Seller information is obtained from the order items table because seller IDs are associated with individual order items rather than directly with the order-level dataset.

Since an order may contain items from multiple sellers, the seller information must be aggregated at the order level before being merged with the train, validation, and test datasets.


In [25]:
order_items = pd.read_csv("../Data/raw/olist_order_items_dataset.csv")

In [26]:
seller_per_order = (
    order_items
    .groupby("order_id")["seller_id"]
    .nunique()
)

seller_per_order.value_counts().sort_index()

seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

### Seller Geographic Coordinates

Seller ZIP code prefixes are obtained from the order items and mapped to the seller dataset.

For orders containing multiple sellers, seller locations will be aggregated at the order level so that the final dataset remains one row per order.


In [27]:
order_seller = (
    order_items[
        ["order_id", "seller_id"]
    ]
    .drop_duplicates()
    .merge(
        sellers[
            ["seller_id", "seller_zip_code_prefix"]
        ],
        on="seller_id",
        how="left"
    )
)

In [28]:
order_seller.head()

,order_id,seller_id,seller_zip_code_prefix
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900


In [29]:
order_seller = order_seller.merge(
    geo_by_zip,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

In [30]:
order_seller[
    ["seller_zip_code_prefix", "latitude", "longitude"]
].isna().sum()

seller_zip_code_prefix      0
latitude                  220
longitude                 220
dtype: int64

In [31]:
order_seller = order_seller.rename(
    columns={
        "latitude": "seller_latitude",
        "longitude": "seller_longitude"
    }
)

In [32]:
order_seller.columns.tolist()

['order_id',
 'seller_id',
 'seller_zip_code_prefix',
 'geolocation_zip_code_prefix',
 'seller_latitude',
 'seller_longitude']

In [33]:
train_seller = train[
    ["order_id", "customer_latitude", "customer_longitude"]
].merge(
    order_seller[
        ["order_id", "seller_latitude", "seller_longitude"]
    ],
    on="order_id",
    how="left"
)

In [34]:
train_seller.shape

(68347, 5)

### Customer-Seller Distance

The geographical distance between the customer and seller locations is calculated using the Haversine formula. This provides an approximate great-circle distance in kilometers based on the latitude and longitude coordinates.

The distance is calculated using only customer and seller location information available at prediction time.


In [35]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

In [36]:
train_seller["distance_km"] = haversine_distance(
    train_seller["customer_latitude"],
    train_seller["customer_longitude"],
    train_seller["seller_latitude"],
    train_seller["seller_longitude"]
)

In [37]:
train_seller["distance_km"].describe()

count    67997.000000
mean       614.267597
std        594.118591
min          0.000000
25%        219.962081
50%        448.427134
75%        810.937579
max       5338.619521
Name: distance_km, dtype: float64

### Aggregating Distance at Order Level

Some orders contain items from multiple sellers. To preserve the one-row-per-order structure required by the machine learning dataset, the customer-seller distances are aggregated by order.

The mean distance across the sellers associated with each order is used as the final `distance_km` feature.


In [38]:
train_distance = (
    train_seller
    .groupby("order_id", as_index=False)["distance_km"]
    .mean()
)

In [39]:
train_distance.shape

(67533, 2)

In [40]:
train = train.merge(
    train_distance,
    on="order_id",
    how="left"
)

In [41]:
train["distance_km"].isna().sum()

np.int64(344)

In [42]:
train["distance_km"].describe()

count    67189.000000
mean       614.743167
std        594.760614
min          0.000000
25%        219.962081
50%        448.619468
75%        810.729727
max       5338.619521
Name: distance_km, dtype: float64

In [43]:
validation_seller = validation[
    ["order_id", "customer_latitude", "customer_longitude"]
].merge(
    order_seller[
        ["order_id", "seller_latitude", "seller_longitude"]
    ],
    on="order_id",
    how="left"
)

validation_seller["distance_km"] = haversine_distance(
    validation_seller["customer_latitude"],
    validation_seller["customer_longitude"],
    validation_seller["seller_latitude"],
    validation_seller["seller_longitude"]
)

validation_distance = (
    validation_seller
    .groupby("order_id", as_index=False)["distance_km"]
    .mean()
)

validation = validation.merge(
    validation_distance,
    on="order_id",
    how="left"
)

In [44]:
validation["distance_km"].isna().sum()

np.int64(74)

In [45]:
test_seller = test[
    ["order_id", "customer_latitude", "customer_longitude"]
].merge(
    order_seller[
        ["order_id", "seller_latitude", "seller_longitude"]
    ],
    on="order_id",
    how="left"
)

test_seller["distance_km"] = haversine_distance(
    test_seller["customer_latitude"],
    test_seller["customer_longitude"],
    test_seller["seller_latitude"],
    test_seller["seller_longitude"]
)

test_distance = (
    test_seller
    .groupby("order_id", as_index=False)["distance_km"]
    .mean()
)

test = test.merge(
    test_distance,
    on="order_id",
    how="left"
)

In [46]:
test["distance_km"].isna().sum()

np.int64(58)

In [47]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 25)
Validation: (14471, 25)
Test: (14472, 25)


In [48]:
train.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'number_of_items',
 'total_price',
 'total_freight',
 'number_of_payments',
 'total_payment_value',
 'delivery_label',
 'purchase_hour',
 'purchase_weekday',
 'purchase_month',
 'estimated_delivery_days',
 'is_holiday',
 'customer_state',
 'customer_zip_code_prefix',
 'geolocation_zip_code_prefix',
 'customer_latitude',
 'customer_longitude',
 'distance_km']

In [49]:
columns_to_drop = [
    "customer_zip_code_prefix",
    "geolocation_zip_code_prefix",
    "customer_latitude",
    "customer_longitude"
]

train = train.drop(columns=columns_to_drop)
validation = validation.drop(columns=columns_to_drop)
test = test.drop(columns=columns_to_drop)

In [50]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 21)
Validation: (14471, 21)
Test: (14472, 21)


## Feature Selection

Based on the EDA findings, the final feature set includes numerical, categorical, temporal, holiday, and geographical features.

The selected features are:

* Order characteristics: `number_of_items`, `total_price`, `total_freight`, `number_of_payments`, `total_payment_value`
* Purchase timing: `purchase_hour`, `purchase_weekday`, `purchase_month`
* Estimated delivery: `estimated_delivery_days`
* Holiday indicator: `is_holiday`
* Customer geography: `customer_state`
* Shipping distance: `distance_km`

Identifiers, target variables, raw timestamps, and variables containing information that would only be available after the prediction point are excluded to prevent data leakage.


In [51]:
feature_cols = [
    "number_of_items",
    "total_price",
    "total_freight",
    "number_of_payments",
    "total_payment_value",
    "purchase_hour",
    "purchase_weekday",
    "purchase_month",
    "estimated_delivery_days",
    "is_holiday",
    "customer_state",
    "distance_km"
]

In [52]:
len(feature_cols), feature_cols

(12,
 ['number_of_items',
  'total_price',
  'total_freight',
  'number_of_payments',
  'total_payment_value',
  'purchase_hour',
  'purchase_weekday',
  'purchase_month',
  'estimated_delivery_days',
  'is_holiday',
  'customer_state',
  'distance_km'])

### Separating Features and Target

The selected features are separated from the target variable before preprocessing.

`X` contains the input features used by the machine learning model, while `y` contains the target variable, `delivery_label`, which indicates whether an order was delivered late or on time.

In [53]:
X_train = train[feature_cols]
y_train = train["delivery_label"]

X_validation = validation[feature_cols]
y_validation = validation["delivery_label"]

X_test = test[feature_cols]
y_test = test["delivery_label"]

In [54]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (67533, 12)
y_train: (67533,)
X_validation: (14471, 12)
y_validation: (14471,)
X_test: (14472, 12)
y_test: (14472,)


### Inspecting Feature Types and Missing Values

Before applying preprocessing, the selected features are inspected to identify their data types and missing values.

This allows the appropriate transformation to be applied to each feature group.

In [55]:
X_train.dtypes

number_of_items            float64
total_price                float64
total_freight              float64
number_of_payments         float64
total_payment_value        float64
purchase_hour                int32
purchase_weekday               str
purchase_month                 str
estimated_delivery_days    float64
is_holiday                    bool
customer_state                 str
distance_km                float64
dtype: object

In [56]:
X_train.isna().sum()

number_of_items              0
total_price                  0
total_freight                0
number_of_payments           1
total_payment_value          1
purchase_hour                0
purchase_weekday             0
purchase_month               0
estimated_delivery_days      0
is_holiday                   0
customer_state               0
distance_km                344
dtype: int64

### Defining Feature Groups

The selected features are divided into numerical and categorical groups so that each group can receive the appropriate preprocessing.

Numerical features will be imputed using statistics learned from the training data. Categorical features will be encoded using an encoder fitted only on the training split.

In [57]:
numeric_features = [
    "number_of_items",
    "total_price",
    "total_freight",
    "number_of_payments",
    "total_payment_value",
    "purchase_hour",
    "estimated_delivery_days",
    "distance_km"
]

categorical_features = [
    "purchase_weekday",
    "purchase_month",
    "customer_state"
]

In [58]:
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 8
Categorical features: 3


### Handling Missing Numerical Values

Missing values in numerical features are handled using median imputation.

The imputer is fitted only on the training data to prevent data leakage. The learned median values are then applied to the validation and test sets.

In [59]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(strategy="median")

In [61]:
numeric_features = [
    "number_of_items",
    "total_price",
    "total_freight",
    "number_of_payments",
    "total_payment_value",
    "purchase_hour",
    "estimated_delivery_days",
    "distance_km"
]

numeric_imputer.fit(X_train[numeric_features])

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['number_of_items','total_price','total_freight',...,'purchase_hour', 'estimated_delivery_days','distance_km']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](8,)","[ 1. , 85. , 16.79,..., 15. , 23.62,448.62]"


In [62]:
X_train[numeric_features] = numeric_imputer.transform(
    X_train[numeric_features]
)

In [63]:
X_validation[numeric_features] = numeric_imputer.transform(
    X_validation[numeric_features]
)

X_test[numeric_features] = numeric_imputer.transform(
    X_test[numeric_features]
)

In [64]:
X_train[numeric_features].isna().sum()

number_of_items            0
total_price                0
total_freight              0
number_of_payments         0
total_payment_value        0
purchase_hour              0
estimated_delivery_days    0
distance_km                0
dtype: int64

In [65]:
X_validation[numeric_features].isna().sum()

number_of_items            0
total_price                0
total_freight              0
number_of_payments         0
total_payment_value        0
purchase_hour              0
estimated_delivery_days    0
distance_km                0
dtype: int64

In [66]:
X_test[numeric_features].isna().sum()

number_of_items            0
total_price                0
total_freight              0
number_of_payments         0
total_payment_value        0
purchase_hour              0
estimated_delivery_days    0
distance_km                0
dtype: int64

In [67]:
from sklearn.preprocessing import OneHotEncoder

In [68]:
categorical_features = [
    "purchase_weekday",
    "purchase_month",
    "customer_state"
]

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [69]:
categorical_encoder.fit(X_train[categorical_features])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [70]:
X_train_encoded = categorical_encoder.transform(
    X_train[categorical_features]
)

In [71]:
X_train_encoded.shape

(67533, 53)

In [72]:
X_validation_encoded = categorical_encoder.transform(
    X_validation[categorical_features]
)

X_test_encoded = categorical_encoder.transform(
    X_test[categorical_features]
)

In [73]:
print("Train:", X_train_encoded.shape)
print("Validation:", X_validation_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (67533, 53)
Validation: (14471, 53)
Test: (14472, 53)


In [74]:
encoded_feature_names = categorical_encoder.get_feature_names_out(
    categorical_features
)

encoded_feature_names

array(['purchase_weekday_Friday', 'purchase_weekday_Monday',
       'purchase_weekday_Saturday', 'purchase_weekday_Sunday',
       'purchase_weekday_Thursday', 'purchase_weekday_Tuesday',
       'purchase_weekday_Wednesday', 'purchase_month_2016-09',
       'purchase_month_2016-10', 'purchase_month_2016-12',
       'purchase_month_2017-01', 'purchase_month_2017-02',
       'purchase_month_2017-03', 'purchase_month_2017-04',
       'purchase_month_2017-05', 'purchase_month_2017-06',
       'purchase_month_2017-07', 'purchase_month_2017-08',
       'purchase_month_2017-09', 'purchase_month_2017-10',
       'purchase_month_2017-11', 'purchase_month_2017-12',
       'purchase_month_2018-01', 'purchase_month_2018-02',
       'purchase_month_2018-03', 'purchase_month_2018-04',
       'customer_state_AC', 'customer_state_AL', 'customer_state_AM',
       'customer_state_AP', 'customer_state_BA', 'customer_state_CE',
       'customer_state_DF', 'customer_state_ES', 'customer_state_GO',
       '

In [75]:
X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)
X_train_encoded_df.shape

(67533, 53)

In [76]:
X_validation_encoded_df = pd.DataFrame(
    X_validation_encoded,
    columns=encoded_feature_names,
    index=X_validation.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

In [77]:
print("Train:", X_train_encoded_df.shape)
print("Validation:", X_validation_encoded_df.shape)
print("Test:", X_test_encoded_df.shape)

Train: (67533, 53)
Validation: (14471, 53)
Test: (14472, 53)


In [78]:
X_train_final = pd.concat(
    [
        X_train[numeric_features],
        X_train_encoded_df
    ],
    axis=1
)
X_train_final.shape

(67533, 61)

In [79]:
X_validation_final = pd.concat(
    [
        X_validation[numeric_features],
        X_validation_encoded_df
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        X_test[numeric_features],
        X_test_encoded_df
    ],
    axis=1
)

In [80]:
print("X_train:", X_train_final.shape)
print("X_validation:", X_validation_final.shape)
print("X_test:", X_test_final.shape)

X_train: (67533, 61)
X_validation: (14471, 61)
X_test: (14472, 61)


In [81]:
print("Train missing:", X_train_final.isna().sum().sum())
print("Validation missing:", X_validation_final.isna().sum().sum())
print("Test missing:", X_test_final.isna().sum().sum())

Train missing: 0
Validation missing: 0
Test missing: 0


In [82]:
print(
    X_train_final.columns.equals(X_validation_final.columns)
)

print(
    X_train_final.columns.equals(X_test_final.columns)
)

True
True


## Data Splitting and Preprocessing

The dataset was divided into three subsets: training, validation, and test sets.

* **Training set:** 67,533 rows
* **Validation set:** 14,471 rows
* **Test set:** 14,472 rows

Feature engineering was performed to create additional features related to purchase timing, estimated delivery duration, holidays, customer location, and geographic distance.

The final feature set consists of **61 features**:

* **8 numerical features**
* **53 one-hot encoded categorical features**

Missing values in numerical features were handled using median imputation, with the imputation statistics learned from the training set only. Categorical variables were transformed using One-Hot Encoding, with unknown categories handled safely in the validation and test sets.

The resulting datasets are ready for model training and evaluation.


In [83]:
y_train.value_counts()

delivery_label
on_time    62242
late        5291
Name: count, dtype: int64

In [84]:
y_train.value_counts(normalize=True) * 100

delivery_label
on_time    92.165312
late        7.834688
Name: proportion, dtype: float64

In [85]:
y_train_binary = (y_train == "late").astype(int)
y_validation_binary = (y_validation == "late").astype(int)
y_test_binary = (y_test == "late").astype(int)

In [86]:
print(y_train_binary.value_counts())

delivery_label
0    62242
1     5291
Name: count, dtype: int64


In [87]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_numeric_scaled = scaler.fit_transform(
    X_train[numeric_features]
)

X_validation_numeric_scaled = scaler.transform(
    X_validation[numeric_features]
)

X_test_numeric_scaled = scaler.transform(
    X_test[numeric_features]
)

In [88]:
print(X_train_numeric_scaled.shape)
print(X_validation_numeric_scaled.shape)
print(X_test_numeric_scaled.shape)

(67533, 8)
(14471, 8)
(14472, 8)


In [89]:
X_train_numeric_scaled_df = pd.DataFrame(
    X_train_numeric_scaled,
    columns=numeric_features,
    index=X_train.index
)

X_validation_numeric_scaled_df = pd.DataFrame(
    X_validation_numeric_scaled,
    columns=numeric_features,
    index=X_validation.index
)

X_test_numeric_scaled_df = pd.DataFrame(
    X_test_numeric_scaled,
    columns=numeric_features,
    index=X_test.index
)

In [90]:
X_train_final = pd.concat(
    [X_train_numeric_scaled_df, X_train_encoded_df],
    axis=1
)

X_validation_final = pd.concat(
    [X_validation_numeric_scaled_df, X_validation_encoded_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test_numeric_scaled_df, X_test_encoded_df],
    axis=1
)

In [91]:
print("Train:", X_train_final.shape)
print("Validation:", X_validation_final.shape)
print("Test:", X_test_final.shape)

Train: (67533, 61)
Validation: (14471, 61)
Test: (14472, 61)


In [92]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_final,
    y_train_binary
)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [93]:
y_validation_pred = logistic_model.predict(X_validation_final)

print(y_validation_pred.shape)

(14471,)


In [94]:
from sklearn.metrics import classification_report

print(classification_report(
    y_validation_binary,
    y_validation_pred,
    target_names=["on_time", "late"]
))

              precision    recall  f1-score   support

     on_time       0.96      1.00      0.98     13847
        late       0.00      0.00      0.00       624

    accuracy                           0.96     14471
   macro avg       0.48      0.50      0.49     14471
weighted avg       0.92      0.96      0.94     14471



c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [95]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_validation_binary, y_validation_pred)

print(cm)

[[13847     0]
 [  624     0]]


In [96]:
logistic_model_balanced = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

logistic_model_balanced.fit(X_train_final, y_train_binary)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [97]:
y_validation_pred_balanced = logistic_model_balanced.predict(
    X_validation_final
)

print(classification_report(
    y_validation_binary,
    y_validation_pred_balanced,
    target_names=["on_time", "late"]
))

              precision    recall  f1-score   support

     on_time       0.97      0.85      0.91     13847
        late       0.12      0.45      0.19       624

    accuracy                           0.83     14471
   macro avg       0.54      0.65      0.55     14471
weighted avg       0.93      0.83      0.87     14471



In [98]:
cm_balanced = confusion_matrix(
    y_validation_binary,
    y_validation_pred_balanced
)

print(cm_balanced)

[[11728  2119]
 [  341   283]]


In [99]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train_final, y_train_binary)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [100]:
y_validation_pred_rf = rf_model.predict(X_validation_final)

print(classification_report(
    y_validation_binary,
    y_validation_pred_rf,
    target_names=["on_time", "late"]
))

              precision    recall  f1-score   support

     on_time       0.96      1.00      0.98     13847
        late       0.28      0.02      0.04       624

    accuracy                           0.96     14471
   macro avg       0.62      0.51      0.51     14471
weighted avg       0.93      0.96      0.94     14471



In [101]:
y_validation_proba = logistic_model_balanced.predict_proba(
    X_validation_final
)[:, 1]

print(y_validation_proba[:10])

[0.52943028 0.36267135 0.33732295 0.18019135 0.31385709 0.18272219
 0.25086896 0.19984142 0.34085394 0.23081048]


In [102]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

for threshold in thresholds:
    y_pred_threshold = (y_validation_proba >= threshold).astype(int)

    precision = precision_score(y_validation_binary, y_pred_threshold)
    recall = recall_score(y_validation_binary, y_pred_threshold)
    f1 = f1_score(y_validation_binary, y_pred_threshold)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.20 | Precision: 0.049 | Recall: 0.990 | F1: 0.094
Threshold: 0.25 | Precision: 0.057 | Recall: 0.973 | F1: 0.107
Threshold: 0.30 | Precision: 0.066 | Recall: 0.921 | F1: 0.124
Threshold: 0.35 | Precision: 0.075 | Recall: 0.803 | F1: 0.138
Threshold: 0.40 | Precision: 0.090 | Recall: 0.678 | F1: 0.159
Threshold: 0.45 | Precision: 0.104 | Recall: 0.553 | F1: 0.175
Threshold: 0.50 | Precision: 0.118 | Recall: 0.454 | F1: 0.187


In [103]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_final,
    y_train_binary
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [104]:
y_validation_pred_xgb = xgb_model.predict(X_validation_final)

print(classification_report(
    y_validation_binary,
    y_validation_pred_xgb,
    target_names=["on_time", "late"]
))

              precision    recall  f1-score   support

     on_time       0.96      1.00      0.98     13847
        late       1.00      0.00      0.00       624

    accuracy                           0.96     14471
   macro avg       0.98      0.50      0.49     14471
weighted avg       0.96      0.96      0.94     14471



In [105]:
print(confusion_matrix(
    y_validation_binary,
    y_validation_pred_xgb
))

[[13847     0]
 [  623     1]]


In [106]:
y_validation_proba_xgb = xgb_model.predict_proba(
    X_validation_final
)[:, 1]

print(y_validation_proba_xgb[:10])

[0.11500574 0.04211962 0.05457743 0.02895068 0.0442219  0.01182588
 0.03436193 0.02037082 0.02901563 0.03242109]


In [107]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

for threshold in thresholds:
    y_pred = (y_validation_proba_xgb >= threshold).astype(int)

    precision = precision_score(y_validation_binary, y_pred, zero_division=0)
    recall = recall_score(y_validation_binary, y_pred, zero_division=0)
    f1 = f1_score(y_validation_binary, y_pred, zero_division=0)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.05 | Precision: 0.088 | Recall: 0.668 | F1: 0.156
Threshold: 0.10 | Precision: 0.162 | Recall: 0.329 | F1: 0.217
Threshold: 0.15 | Precision: 0.206 | Recall: 0.141 | F1: 0.167
Threshold: 0.20 | Precision: 0.258 | Recall: 0.062 | F1: 0.101
Threshold: 0.25 | Precision: 0.203 | Recall: 0.019 | F1: 0.035
Threshold: 0.30 | Precision: 0.147 | Recall: 0.008 | F1: 0.015
Threshold: 0.35 | Precision: 0.133 | Recall: 0.003 | F1: 0.006
Threshold: 0.40 | Precision: 0.100 | Recall: 0.002 | F1: 0.003
Threshold: 0.45 | Precision: 0.125 | Recall: 0.002 | F1: 0.003
Threshold: 0.50 | Precision: 1.000 | Recall: 0.002 | F1: 0.003


In [108]:
xgb_threshold = 0.10

y_validation_pred_xgb_threshold = (
    y_validation_proba_xgb >= xgb_threshold
).astype(int)

print(classification_report(
    y_validation_binary,
    y_validation_pred_xgb_threshold,
    target_names=["on_time", "late"],
    zero_division=0
))

print(confusion_matrix(
    y_validation_binary,
    y_validation_pred_xgb_threshold
))

              precision    recall  f1-score   support

     on_time       0.97      0.92      0.95     13847
        late       0.16      0.33      0.22       624

    accuracy                           0.90     14471
   macro avg       0.57      0.63      0.58     14471
weighted avg       0.93      0.90      0.91     14471

[[12787  1060]
 [  419   205]]


In [109]:
xgb_threshold_05 = 0.05

y_validation_pred_xgb_05 = (
    y_validation_proba_xgb >= xgb_threshold_05
).astype(int)

print(classification_report(
    y_validation_binary,
    y_validation_pred_xgb_05,
    target_names=["on_time", "late"],
    zero_division=0
))

print(confusion_matrix(
    y_validation_binary,
    y_validation_pred_xgb_05
))

              precision    recall  f1-score   support

     on_time       0.98      0.69      0.81     13847
        late       0.09      0.67      0.16       624

    accuracy                           0.69     14471
   macro avg       0.53      0.68      0.48     14471
weighted avg       0.94      0.69      0.78     14471

[[9530 4317]
 [ 207  417]]


## Model Selection

Logistic Regression and XGBoost were evaluated using the validation set. XGBoost was selected as the final model because it achieved the highest F1-score for the `late` class.

The decision threshold was set to `0.10` based on the validation results to achieve the best balance between precision and recall for late delivery prediction.

In [110]:
y_test_proba_xgb = xgb_model.predict_proba(X_test_final)[:, 1]

y_test_pred_xgb = (
    y_test_proba_xgb >= 0.10
).astype(int)

print(classification_report(
    y_test_binary,
    y_test_pred_xgb,
    target_names=["on_time", "late"],
    zero_division=0
))

print(confusion_matrix(
    y_test_binary,
    y_test_pred_xgb
))

              precision    recall  f1-score   support

     on_time       0.96      0.82      0.89     13852
        late       0.07      0.31      0.12       620

    accuracy                           0.80     14472
   macro avg       0.52      0.56      0.50     14472
weighted avg       0.93      0.80      0.85     14472

[[11358  2494]
 [  430   190]]


In [111]:
import joblib
import os

os.makedirs("../Artifacts/models", exist_ok=True)

joblib.dump(
    xgb_model,
    "../Artifacts/models/xgboost_model.joblib"
)

['../Artifacts/models/xgboost_model.joblib']

In [112]:
joblib.dump(
    numeric_imputer,
    "../Artifacts/models/numeric_imputer.joblib"
)

joblib.dump(
    categorical_encoder,
    "../Artifacts/models/categorical_encoder.joblib"
)

joblib.dump(
    scaler,
    "../Artifacts/models/scaler.joblib"
)

['../Artifacts/models/scaler.joblib']

In [113]:
os.listdir("../Artifacts/models")

['categorical_encoder.joblib',
 'numeric_imputer.joblib',
 'scaler.joblib',
 'xgboost_model.joblib']

## Conclusion

In this notebook, we prepared the Olist dataset for machine learning and developed models to predict whether an order would be delivered on time or late.

The preprocessing process included handling missing values, encoding categorical features, scaling numerical features, and splitting the data into training, validation, and test sets.

Multiple models were evaluated, with particular attention to the imbalanced nature of the target variable. Threshold tuning was also explored to improve the detection of late deliveries.

The final selected XGBoost model was evaluated on the unseen test set using a threshold of 0.10. The model achieved a recall of approximately 31% for late deliveries, correctly identifying 190 out of 620 late orders. The overall accuracy was approximately 80%.

Although the model's ability to detect late deliveries remains limited, recall for the late class was prioritized because late deliveries represent a minority class and are the primary prediction target.

The trained XGBoost model and preprocessing components were saved as reusable artifacts for future inference and deployment.
